In [1]:
from pathlib import Path
import sys

root = Path.cwd()
while not (root / "src").is_dir() and root != root.parent:
    root = root.parent

if not (root / "src").is_dir():
    raise FileNotFoundError("Could not find the repository folder containing src/")

sys.path.insert(0, str(root))

In [2]:
import numpy as np
import pandas as pd
from qiskit.circuit.library import ZZFeatureMap
from qiskit_machine_learning.kernels import FidelityQuantumKernel
from sklearn.preprocessing import MinMaxScaler
from sklearn.svm import SVC
from sklearn.metrics import accuracy_score, recall_score, f1_score
from src.data_load import load_data
from src.preprocess import preprocess

In [5]:
qubits = 4
reps = 2
entanglement = "linear"
ranges = [np.pi, 1.0, 0.5, 0.25]      
C_values = [0.1, 1, 10, 100]

X_train, X_test, y_train, y_test = load_data()
_, _, X_train_pca, X_test_pca, pca = preprocess(X_train, X_test, qubits)

In [6]:
feature_map = ZZFeatureMap(feature_dimension=qubits, reps=reps, entanglement=entanglement)
kernel = FidelityQuantumKernel(feature_map=feature_map)

C:\Users\nonym\AppData\Local\Temp\ipykernel_35320\653730057.py:1: DeprecationWarning: The class ``qiskit.circuit.library.data_preparation._zz_feature_map.ZZFeatureMap`` is deprecated as of Qiskit 2.1. It will be removed in Qiskit 3.0. Use the zz_feature_map function as a replacement. Note that this will no longer return a BlueprintCircuit, but just a plain QuantumCircuit.
  feature_map = ZZFeatureMap(feature_dimension=qubits, reps=reps, entanglement=entanglement)


In [7]:
rows = []
for upper in ranges:
    mm = MinMaxScaler(feature_range=(0, upper))
    Xtr = mm.fit_transform(X_train_pca)
    Xte = np.clip(mm.transform(X_test_pca), 0, upper)

    K_train = kernel.evaluate(Xtr, Xtr)
    K_test = kernel.evaluate(Xte, Xtr)

    off = K_train[~np.eye(len(K_train), dtype=bool)]
    np.save(f"K_train_zz{qubits}_reps{reps}_{entanglement}_r{upper:.2f}.npy", K_train)
    np.save(f"K_test_zz{qubits}_reps{reps}_{entanglement}_r{upper:.2f}.npy", K_test)

    for C in C_values:
        model = SVC(kernel="precomputed", C=C).fit(K_train, y_train)
        pred = model.predict(K_test)
        rows.append({
            "range_upper": round(upper, 2),
            "C": C,
            "accuracy": accuracy_score(y_test, pred),
            "malignant_recall": recall_score(y_test, pred, pos_label=0),
            "f1_macro": f1_score(y_test, pred, average="macro"),
            "kernel_mean": off.mean(),
            "kernel_std": off.std(),
        })
    print(f"done: range (0, {upper:.2f})")

done: range (0, 3.14)
done: range (0, 1.00)
done: range (0, 0.50)
done: range (0, 0.25)


In [8]:
results = pd.DataFrame(rows)
results.to_csv("zz_range_sweep.csv", index=False)
print(results.to_string(index=False))

 range_upper     C  accuracy  malignant_recall  f1_macro  kernel_mean  kernel_std
        3.14   0.1  0.666667          0.095238  0.482561     0.083659    0.109280
        3.14   1.0  0.859649          0.761905  0.845946     0.083659    0.109280
        3.14  10.0  0.850877          0.833333  0.842015     0.083659    0.109280
        3.14 100.0  0.842105          0.833333  0.833442     0.083659    0.109280
        1.00   0.1  0.921053          0.880952  0.914749     0.214618    0.240627
        1.00   1.0  0.929825          0.904762  0.924603     0.214618    0.240627
        1.00  10.0  0.938596          0.904762  0.933693     0.214618    0.240627
        1.00 100.0  0.921053          0.904762  0.915590     0.214618    0.240627
        0.50   0.1  0.912281          0.833333  0.903716     0.519415    0.282051
        0.50   1.0  0.947368          0.928571  0.943452     0.519415    0.282051
        0.50  10.0  0.947368          0.928571  0.943452     0.519415    0.282051
        0.50 100